In [1]:
import numpyro
import numpyro.distributions as dist
import jax
import jax.numpy as jnp
import numpy as np

print(f"NumPyro version : {numpyro.__version__}")
print(f"JAX version     : {jax.__version__}")
print(f"JAX backend     : {jax.default_backend()}")

NumPyro version : 0.21.0
JAX version     : 0.10.0
JAX backend     : cpu


In [3]:
# =============================================================================
# LEAGUE-LEVEL PRIORS
# Prior spec section 1.1, 1.2, 1.3
# All parameters are additive on the log scale — this is a log-linear model.
# =============================================================================

def prior_specification():

    # -------------------------------------------------------------------------
    # 1.1 League intercept
    # Log scale: exp(3.3) ≈ 27 pts (FBS mean pts/team/game 2022-2024)
    # SD=0.2 allows posterior to move while ruling out implausible baselines
    # Prior spec section 1.1
    # -------------------------------------------------------------------------
    mu_league = numpyro.sample("mu_league", dist.Normal(3.3, 0.2))

    # -------------------------------------------------------------------------
    # 1.2 Home field advantage baseline
    # Log scale: 0.1 ≈ 2.5 pts on a 27 pt baseline (2.5/27)
    # EDA 05: league-level HFA = +2.43 pts (p<0.001)
    # No conference-level HFA layer — team deviations handle within-conf variation
    # Prior spec section 1.2
    # -------------------------------------------------------------------------
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))

    # -------------------------------------------------------------------------
    # 1.3 Dispersion parameter — conference-specific vector
    # EDA 05: Bartlett p=0.000470, Levene p=0.000705 — conference dispersion
    #   differences are statistically confirmed, not just assumed
    # EDA 01: VMR range 4.95-7.16 across conferences — scalar r cannot
    #   capture this variation; scalar HalfNormal(5.0) caused sampler collapse
    # r_negbinom is a VECTOR of length N_CONFERENCES, one per conference
    # Prior: Gamma(2.0, 0.1) — mean=20, SD≈14 on r scale, weakly informative
    #   Gamma(2.0, 0.1) uses shape=2.0, rate=0.1 (numpyro Gamma parameterization)
    # Each conference independently — NOT sharing a hyperprior
    # Prior spec section 1.3
    # -------------------------------------------------------------------------
    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(2.0, 0.1),
        sample_shape=(N_CONFERENCES,)
    )

    return mu_league, hfa_league, r_negbinom

print("League-level priors defined:")
print(f"  mu_league   : Normal(3.3, 0.2)   [log scale; exp(3.3) ≈ 27 pts]")
print(f"  hfa_league  : Normal(0.1, 0.05)  [log scale; ≈ 2.43 pts on 27 pt baseline]")
print(f"  r_negbinom  : Gamma(2.0, 0.1) x N_CONFERENCES conferences  [conference-specific dispersion]")
print("Cell 2 complete.")

League-level priors defined:
  mu_league   : Normal(3.3, 0.2)   [log scale; exp(3.3) ≈ 27 pts]
  hfa_league  : Normal(0.1, 0.05)  [log scale; ≈ 2.43 pts on 27 pt baseline]
  r_negbinom  : Gamma(2.0, 0.1) x N_CONFERENCES conferences  [conference-specific dispersion]
Cell 2 complete.


In [4]:
# =============================================================================
# CONFERENCE-LEVEL PRIORS
# Prior spec section 2.1
# =============================================================================

# 10 FBS conferences in training data
CONFERENCES = [
    "ACC", "American Athletic", "Big 12", "Big Ten",
    "Conference USA", "Mid-American", "Mountain West",
    "Pac-12", "SEC", "Sun Belt"
]
N_CONFERENCES = len(CONFERENCES)

# Conference index lookup — used throughout all subsequent notebooks
conf_to_idx = {c: i for i, c in enumerate(CONFERENCES)}

def add_conference_priors():
    # -------------------------------------------------------------------------
    # 2.1 Conference scoring offset hyperprior
    # Day 10: conference ICC marginal (0.02-0.05) but pooling improves
    # small-sample estimates
    # Hyperprior SD=3.0 allows realistic conf-level scoring differences
    # while regularizing toward league mean
    # Offset from league baseline (mu_league), not absolute scoring level
    # Prior spec section 2.1
    # -------------------------------------------------------------------------
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(3.0))
    mu_conference = numpyro.sample(
        "mu_conference",
        dist.Normal(0.0, sigma_conference),
        sample_shape=(N_CONFERENCES,)
    )
    return sigma_conference, mu_conference

print("Conference-level priors defined:")
print(f"  sigma_conference : HalfNormal(3.0)  [hyperprior SD]")
print(f"  mu_conference    : Normal(0.0, sigma_conference) x {N_CONFERENCES} conferences")
print(f"\nConference index map:")
for c, i in conf_to_idx.items():
    print(f"  {i:2d} : {c}")
print("Cell 3 complete.")

Conference-level priors defined:
  sigma_conference : HalfNormal(3.0)  [hyperprior SD]
  mu_conference    : Normal(0.0, sigma_conference) x 10 conferences

Conference index map:
   0 : ACC
   1 : American Athletic
   2 : Big 12
   3 : Big Ten
   4 : Conference USA
   5 : Mid-American
   6 : Mountain West
   7 : Pac-12
   8 : SEC
   9 : Sun Belt
Cell 3 complete.


In [5]:
# =============================================================================
# TEAM-LEVEL PRIORS
# Prior spec section 3.1, 3.2, 3.3, 3.4, 3.5
# =============================================================================

# Placeholder — will be replaced with actual team count from training data
# when model_04 loads the database. Set here for prior specification purposes.
N_TEAMS = 130  # approximate FBS teams x 3 seasons yields ~130 unique teams

def add_team_priors():
    # -------------------------------------------------------------------------
    # 3.1 Team attack parameter (log-scale offensive strength)
    # EDA 05: team ICC for points_scored = 0.1394 — justifies team-level params
    # YoY r for raw scoring 0.35-0.49 — too unstable to use directly
    # Prior anchored by SP+ and EPA instead
    # Offset from conference baseline
    # Prior spec section 3.1
    # -------------------------------------------------------------------------
    sigma_attack = numpyro.sample("sigma_attack", dist.HalfNormal(0.4))
    alpha_team = numpyro.sample(
        "alpha_team",
        dist.Normal(0.0, sigma_attack),
        sample_shape=(N_TEAMS,)
    )

    # -------------------------------------------------------------------------
    # 3.2 Team defense parameter (log-scale defensive strength)
    # EDA 05: team ICC for point_differential = 0.1925 — strongest ICC finding
    # Symmetric with attack parameter
    # Offset from conference baseline
    # Prior spec section 3.2
    # -------------------------------------------------------------------------
    sigma_defense = numpyro.sample("sigma_defense", dist.HalfNormal(0.4))
    delta_team = numpyro.sample(
        "delta_team",
        dist.Normal(0.0, sigma_defense),
        sample_shape=(N_TEAMS,)
    )

    # -------------------------------------------------------------------------
    # 3.3 Team home field deviation
    # Log scale: HalfNormal(0.1) allows realistic team-level HFA variation
    # while regularizing sparse teams toward league baseline
    # EDA 05: team HFA SD = 4.81 pts; 4.81/27 ≈ 0.18 on log scale
    # Deviation from league hfa_league, not absolute HFA
    # Prior spec section 3.3
    # -------------------------------------------------------------------------
    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team = numpyro.sample(
        "hfa_team",
        dist.Normal(0.0, sigma_hfa_team),
        sample_shape=(N_TEAMS,)
    )

    # -------------------------------------------------------------------------
    # 3.4 SP+ prior seed (team-level coefficient)
    # EDA 04: YoY r = 0.7632 — strong stability (corrected from 0.7740)
    # DUAL ROLE: functions as BOTH prior seed AND game-level spread predictor
    #   Partial r = 0.197 after EPA control, p<0.0001, all conferences, all bands
    #   Signal does not decay aggressively — holds through games 9-12 (r=0.2609)
    #   Do NOT treat as prior-only in model_02 architecture
    # SP+ components (sp_offense, sp_defense) excluded — DATA LEAKAGE
    #   (end-of-season values, not collinearity)
    # Prior spec section 3.4
    # -------------------------------------------------------------------------
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 1.0))

    # -------------------------------------------------------------------------
    # 3.5 Recruiting prior seed (conference-specific coefficients)
    # EDA 04: YoY r = 0.9779 — extremely stable
    # Conference-specific weights: Big Ten and SEC moderate; all others low
    # Sun Belt: HARD CONSTRAINT — weight must be non-positive
    #   EDA 04: rec<->diff_r = -0.2665 in Sun Belt; positive weight would
    #   introduce systematic error
    # Implemented as TruncatedNormal(high=0) for Sun Belt index (idx=9)
    # All other conferences: Normal(0.0, 0.5)
    # Prior spec section 3.5
    # -------------------------------------------------------------------------
    SUN_BELT_IDX = conf_to_idx["Sun Belt"]  # index 9

    rec_weight_other = numpyro.sample(
        "rec_weight_other",
        dist.Normal(0.0, 0.5),
        sample_shape=(N_CONFERENCES - 1,)
    )
    rec_weight_sunbelt = numpyro.sample(
        "rec_weight_sunbelt",
        dist.TruncatedNormal(0.0, 0.5, high=0.0)
    )
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    return (sigma_attack, alpha_team, sigma_defense, delta_team,
            sigma_hfa_team, hfa_team, sp_weight, rec_weight)

print("Team-level priors defined:")
print(f"  sigma_attack     : HalfNormal(0.4)")
print(f"  alpha_team       : Normal(0.0, sigma_attack) x {N_TEAMS} teams")
print(f"  sigma_defense    : HalfNormal(0.4)")
print(f"  delta_team       : Normal(0.0, sigma_defense) x {N_TEAMS} teams")
print(f"  sigma_hfa_team   : HalfNormal(0.1)  [log scale; ≈ 4.81 pts team HFA SD]")
print(f"  hfa_team         : Normal(0.0, sigma_hfa_team) x {N_TEAMS} teams")
print(f"  sp_weight        : Normal(0.0, 1.0)  [prior seed + game-level predictor]")
print(f"  rec_weight[0-8]  : Normal(0.0, 0.5)  [non-Sun-Belt conferences]")
print(f"  rec_weight[9]    : TruncatedNormal(0.0, 0.5, high=0.0)  [Sun Belt — hard constraint]")
print("Cell 4 complete.")

Team-level priors defined:
  sigma_attack     : HalfNormal(0.4)
  alpha_team       : Normal(0.0, sigma_attack) x 130 teams
  sigma_defense    : HalfNormal(0.4)
  delta_team       : Normal(0.0, sigma_defense) x 130 teams
  sigma_hfa_team   : HalfNormal(0.1)  [log scale; ≈ 4.81 pts team HFA SD]
  hfa_team         : Normal(0.0, sigma_hfa_team) x 130 teams
  sp_weight        : Normal(0.0, 1.0)  [prior seed + game-level predictor]
  rec_weight[0-8]  : Normal(0.0, 0.5)  [non-Sun-Belt conferences]
  rec_weight[9]    : TruncatedNormal(0.0, 0.5, high=0.0)  [Sun Belt — hard constraint]
Cell 4 complete.


In [6]:
# =============================================================================
# GAME-LEVEL FEATURE PRIORS
# Prior spec sections 4.1 through 4.9
# =============================================================================

def add_game_level_priors():

    # -------------------------------------------------------------------------
    # 4.1 Close-game EPA anchor pair
    # EDA 03: spread partial r = 0.5988 and -0.6134 at conf game 1
    # EDA 03: O/U partial r = 0.4237 and 0.4473
    # EDA 03: YoY r = 0.4331 and 0.4224
    # Signal holds across full season trajectory — anchor status confirmed
    # SD=0.5 allows substantial effect while ruling out implausible coefficients
    # Prior spec section 4.1
    # -------------------------------------------------------------------------
    b_close_game_epa         = numpyro.sample("b_close_game_epa",     dist.Normal(0.0, 0.5))
    b_close_game_def_epa     = numpyro.sample("b_close_game_def_epa", dist.Normal(0.0, 0.5))

    # -------------------------------------------------------------------------
    # 4.2 Pregame ELO
    # EDA 08: spread partial r = 0.1664; conf game 1 r = 0.1819
    # EDA 08: YoY r = 0.8423 — highly stable game-level predictor
    # Spread signal only; O/U signal absent
    # Prior spec section 4.2
    # -------------------------------------------------------------------------
    b_pregame_elo            = numpyro.sample("b_pregame_elo",        dist.Normal(0.0, 0.3))

    # -------------------------------------------------------------------------
    # 4.3 ELO/SP+ divergence
    # EDA 08 (corrected): spread r = -0.1150 after SP+ controlled (NEGATIVE)
    #   Negative direction: ELO-underrated teams outperform SP+ expectation
    #   Pre-fix value r=+0.1650 used raw ELO minus raw SP+ (meaningless scale)
    #   Post-fix: z-score difference, mean=0.000, SD=0.449
    # EDA 08: divergence YoY r = 0.076 — not stable, game-level predictor only
    # Smaller SD=0.2 — interaction-style feature, more regularization warranted
    # Computed in notebook as z-score difference; not a raw dbt column
    # Prior spec section 4.3
    # -------------------------------------------------------------------------
    b_elo_sp_divergence      = numpyro.sample("b_elo_sp_divergence",  dist.Normal(0.0, 0.2))

    # -------------------------------------------------------------------------
    # 4.4 Rolling momentum features
    # EDA 07: conference-specific spread signal from conf game 2 onward
    # Null at conf game 1 — handled by impute_season_prior (Approach A)
    # last3_win_pct applied across all conferences
    # SD=0.3 for all momentum features
    # Prior spec section 4.4
    # -------------------------------------------------------------------------
    b_last3_win_pct          = numpyro.sample("b_last3_win_pct",          dist.Normal(0.0, 0.3))
    b_last3_off_epa_avg      = numpyro.sample("b_last3_off_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_def_epa_avg      = numpyro.sample("b_last3_def_epa_avg",      dist.Normal(0.0, 0.3))
    b_last3_pts_scored_avg   = numpyro.sample("b_last3_pts_scored_avg",   dist.Normal(0.0, 0.3))
    b_last3_pts_allowed_avg  = numpyro.sample("b_last3_pts_allowed_avg",  dist.Normal(0.0, 0.3))

    # -------------------------------------------------------------------------
    # 4.5 Environmental features (threshold-activated)
    # All modeled as indicator x magnitude — value=0 when threshold not met
    # Zeros must remain zero at preprocessing: divide by non-zero std only,
    #   no mean subtraction (per feature preprocessing rules)
    # away_elevation_delta_ft : active >= 2000 ft, EDA 06 YoY r = 0.8258, SD=0.3
    # away_travel_distance_mi : active >= 1500 mi, EDA 06 YoY r = 0.6562, SD=0.3
    # away_tz_delta_hrs       : active abs >= 2 hr, EDA 06 YoY r = 0.6710, SD=0.3
    # wind_chill              : active <= 40F and NOT dome, O/U only, SD=0.2
    # Prior spec section 4.5
    # -------------------------------------------------------------------------
    b_away_elevation_delta   = numpyro.sample("b_away_elevation_delta",   dist.Normal(0.0, 0.3))
    b_away_travel_distance   = numpyro.sample("b_away_travel_distance",   dist.Normal(0.0, 0.3))
    b_away_tz_delta          = numpyro.sample("b_away_tz_delta",          dist.Normal(0.0, 0.3))
    b_wind_chill             = numpyro.sample("b_wind_chill",             dist.Normal(0.0, 0.2))

    # -------------------------------------------------------------------------
    # 4.6 Style/tempo delta features
    # EDA 09: rush_rate_std_downs_delta most consistent (r=0.293 full pop)
    #   Signal holds all season buckets including game 1 (r=0.2965-0.3628)
    # EDA 09: YoY stability weak (r≈0.49/0.46) — game-level predictors only,
    #   not prior seeds
    # DEPLOYMENT CONDITION: all style delta features are in-game metrics;
    #   pregame rolling versions required before 2026-09-24 production launch
    # SD=0.3 for all
    # Prior spec section 4.6
    # -------------------------------------------------------------------------
    b_rush_rate_std_downs    = numpyro.sample("b_rush_rate_std_downs",    dist.Normal(0.0, 0.3))
    b_rush_rate_pass_downs   = numpyro.sample("b_rush_rate_pass_downs",   dist.Normal(0.0, 0.3))

    # -------------------------------------------------------------------------
    # 4.7 Sack-rate mismatch features (moneyline variance)
    # EDA 09: abs residual variance partial r = ±0.0919
    # Affect dispersion component, not the mean
    # Smaller SD=0.2 — weaker signal, more regularization needed
    # Prior spec section 4.7
    # -------------------------------------------------------------------------
    b_off_sack_rate_allowed  = numpyro.sample("b_off_sack_rate_allowed",  dist.Normal(0.0, 0.2))
    b_def_sack_rate          = numpyro.sample("b_def_sack_rate",          dist.Normal(0.0, 0.2))

    # -------------------------------------------------------------------------
    # 4.8 Style archetype matchup features
    # EDA 10: O/U signal confirmed — defense matchup eta²=0.39 (strongest finding)
    # EDA 10: weak secondary spread signal (eta²=0.021); no moneyline variance
    # EDA 10: archetype retention 25-40% YoY — NOT stable for prior seeding
    # ENCODING: off_archetype_id and def_archetype_id are integers 0-3
    #   Each is embedded as a 4-vector: b_off_archetype[archetype_id]
    #   data.archetype_idx must be int32, NOT float32
    #   Do NOT use the 16-way compound matchup string columns
    # DEPLOYMENT CONDITION: pregame-deployable rolling version required
    #   before 2026-09-24 launch; drop at refinement if not available
    # Prior spec section 4.8
    # -------------------------------------------------------------------------
    b_off_archetype          = numpyro.sample("b_off_archetype",
                                               dist.Normal(0.0, 0.3),
                                               sample_shape=(4,))
    b_def_archetype          = numpyro.sample("b_def_archetype",
                                               dist.Normal(0.0, 0.3),
                                               sample_shape=(4,))

    # -------------------------------------------------------------------------
    # 4.9 Days since last game (bye week)
    # EDA 07: bye week signal (>= 12 days) in American Athletic and Big 12 only
    #   AmAth r=-0.105, Big 12 r=+0.126 — opposite directions, thin samples
    #   (n≈43-44). Do not treat as reliably established conference effects.
    # Threshold-activated; zero outside confirmed conferences
    # SD=0.2 — narrow scope, more regularization
    # Prior spec section 4.9
    # -------------------------------------------------------------------------
    b_days_since_last_game   = numpyro.sample("b_days_since_last_game",   dist.Normal(0.0, 0.2))

    return {
        "b_close_game_epa":        b_close_game_epa,
        "b_close_game_def_epa":    b_close_game_def_epa,
        "b_pregame_elo":           b_pregame_elo,
        "b_elo_sp_divergence":     b_elo_sp_divergence,
        "b_last3_win_pct":         b_last3_win_pct,
        "b_last3_off_epa_avg":     b_last3_off_epa_avg,
        "b_last3_def_epa_avg":     b_last3_def_epa_avg,
        "b_last3_pts_scored_avg":  b_last3_pts_scored_avg,
        "b_last3_pts_allowed_avg": b_last3_pts_allowed_avg,
        "b_away_elevation_delta":  b_away_elevation_delta,
        "b_away_travel_distance":  b_away_travel_distance,
        "b_away_tz_delta":         b_away_tz_delta,
        "b_wind_chill":            b_wind_chill,
        "b_rush_rate_std_downs":   b_rush_rate_std_downs,
        "b_rush_rate_pass_downs":  b_rush_rate_pass_downs,
        "b_off_sack_rate_allowed": b_off_sack_rate_allowed,
        "b_def_sack_rate":         b_def_sack_rate,
        "b_off_archetype":         b_off_archetype,
        "b_def_archetype":         b_def_archetype,
        "b_days_since_last_game":  b_days_since_last_game,
    }

print("Game-level feature priors defined:")
print(f"  EPA anchors (x2)          : Normal(0.0, 0.5)")
print(f"  pregame_elo               : Normal(0.0, 0.3)")
print(f"  elo_sp_divergence         : Normal(0.0, 0.2)  [r=-0.1150, negative direction]")
print(f"  momentum features (x5)    : Normal(0.0, 0.3)")
print(f"  environmental (x3)        : Normal(0.0, 0.3)")
print(f"  wind_chill                : Normal(0.0, 0.2)")
print(f"  style/tempo deltas (x2)   : Normal(0.0, 0.3)")
print(f"  sack-rate mismatch (x2)   : Normal(0.0, 0.2)")
print(f"  b_off_archetype           : Normal(0.0, 0.3) x 4  [categorical embedding]")
print(f"  b_def_archetype           : Normal(0.0, 0.3) x 4  [categorical embedding]")
print(f"  days_since_last_game      : Normal(0.0, 0.2)")
print(f"\nTotal game-level coefficient vectors: 20 scalars + 2 embeddings (4-vector each)")
print("Cell 5 complete.")

Game-level feature priors defined:
  EPA anchors (x2)          : Normal(0.0, 0.5)
  pregame_elo               : Normal(0.0, 0.3)
  elo_sp_divergence         : Normal(0.0, 0.2)  [r=-0.1150, negative direction]
  momentum features (x5)    : Normal(0.0, 0.3)
  environmental (x3)        : Normal(0.0, 0.3)
  wind_chill                : Normal(0.0, 0.2)
  style/tempo deltas (x2)   : Normal(0.0, 0.3)
  sack-rate mismatch (x2)   : Normal(0.0, 0.2)
  b_off_archetype           : Normal(0.0, 0.3) x 4  [categorical embedding]
  b_def_archetype           : Normal(0.0, 0.3) x 4  [categorical embedding]
  days_since_last_game      : Normal(0.0, 0.2)

Total game-level coefficient vectors: 20 scalars + 2 embeddings (4-vector each)
Cell 5 complete.


In [7]:
# =============================================================================
# COMPLETE PRIOR SPECIFICATION MODEL
# Assembles all priors from cells 2-5 into a single numpyro model function
# model_01 only — no MCMC.run(), no fitting, no data
# =============================================================================

def model_prior_spec():
    """
    Full prior specification for the hierarchical Negative Binomial CFB model.
    Three-level hierarchy: league -> conference -> team.
    This function defines all priors. It does not fit the model.
    model_02 (architecture) and model_04 (first fit) build on this foundation.
    All parameters are additive on the log scale.
    """

    # --- League level (prior spec sections 1.1, 1.2, 1.3) ---
    # Log scale: exp(3.3) ≈ 27 pts; hfa 0.1 ≈ 2.43 pts on 27 pt baseline
    # r_negbinom: conference-specific vector — scalar caused sampler collapse
    mu_league  = numpyro.sample("mu_league",  dist.Normal(3.3, 0.2))
    hfa_league = numpyro.sample("hfa_league", dist.Normal(0.1, 0.05))
    r_negbinom = numpyro.sample(
        "r_negbinom",
        dist.Gamma(2.0, 0.1),
        sample_shape=(N_CONFERENCES,)
    )

    # --- Conference level (prior spec section 2.1) ---
    sigma_conference = numpyro.sample("sigma_conference", dist.HalfNormal(3.0))
    mu_conference    = numpyro.sample("mu_conference",
                                      dist.Normal(0.0, sigma_conference),
                                      sample_shape=(N_CONFERENCES,))

    # --- Team level (prior spec sections 3.1, 3.2, 3.3) ---
    # Non-centered parameterization used in model_02 for alpha_team, delta_team,
    # hfa_team — defined centered here for prior specification clarity only
    sigma_attack = numpyro.sample("sigma_attack", dist.HalfNormal(0.4))
    alpha_team   = numpyro.sample("alpha_team",
                                   dist.Normal(0.0, sigma_attack),
                                   sample_shape=(N_TEAMS,))

    sigma_defense = numpyro.sample("sigma_defense", dist.HalfNormal(0.4))
    delta_team    = numpyro.sample("delta_team",
                                    dist.Normal(0.0, sigma_defense),
                                    sample_shape=(N_TEAMS,))

    sigma_hfa_team = numpyro.sample("sigma_hfa_team", dist.HalfNormal(0.1))
    hfa_team       = numpyro.sample("hfa_team",
                                     dist.Normal(0.0, sigma_hfa_team),
                                     sample_shape=(N_TEAMS,))

    # --- Prior seeds (prior spec sections 3.4, 3.5) ---
    # sp_weight: BOTH prior seed AND game-level spread predictor (partial r=0.197)
    sp_weight = numpyro.sample("sp_weight", dist.Normal(0.0, 1.0))

    SUN_BELT_IDX       = conf_to_idx["Sun Belt"]
    rec_weight_other   = numpyro.sample("rec_weight_other",
                                         dist.Normal(0.0, 0.5),
                                         sample_shape=(N_CONFERENCES - 1,))
    rec_weight_sunbelt = numpyro.sample("rec_weight_sunbelt",
                                         dist.TruncatedNormal(0.0, 0.5, high=0.0))
    rec_weight = jnp.concatenate([
        rec_weight_other[:SUN_BELT_IDX],
        rec_weight_sunbelt[jnp.newaxis],
        rec_weight_other[SUN_BELT_IDX:]
    ])

    # --- Game-level features (prior spec sections 4.1-4.9) ---
    b_close_game_epa        = numpyro.sample("b_close_game_epa",        dist.Normal(0.0, 0.5))
    b_close_game_def_epa    = numpyro.sample("b_close_game_def_epa",    dist.Normal(0.0, 0.5))
    b_pregame_elo           = numpyro.sample("b_pregame_elo",           dist.Normal(0.0, 0.3))
    # elo_sp_divergence: r=-0.1150 (negative direction, z-score version, EDA 08 corrected)
    b_elo_sp_divergence     = numpyro.sample("b_elo_sp_divergence",     dist.Normal(0.0, 0.2))
    b_last3_win_pct         = numpyro.sample("b_last3_win_pct",         dist.Normal(0.0, 0.3))
    b_last3_off_epa_avg     = numpyro.sample("b_last3_off_epa_avg",     dist.Normal(0.0, 0.3))
    b_last3_def_epa_avg     = numpyro.sample("b_last3_def_epa_avg",     dist.Normal(0.0, 0.3))
    b_last3_pts_scored_avg  = numpyro.sample("b_last3_pts_scored_avg",  dist.Normal(0.0, 0.3))
    b_last3_pts_allowed_avg = numpyro.sample("b_last3_pts_allowed_avg", dist.Normal(0.0, 0.3))
    b_away_elevation_delta  = numpyro.sample("b_away_elevation_delta",  dist.Normal(0.0, 0.3))
    b_away_travel_distance  = numpyro.sample("b_away_travel_distance",  dist.Normal(0.0, 0.3))
    b_away_tz_delta         = numpyro.sample("b_away_tz_delta",         dist.Normal(0.0, 0.3))
    b_wind_chill            = numpyro.sample("b_wind_chill",            dist.Normal(0.0, 0.2))
    b_rush_rate_std_downs   = numpyro.sample("b_rush_rate_std_downs",   dist.Normal(0.0, 0.3))
    b_rush_rate_pass_downs  = numpyro.sample("b_rush_rate_pass_downs",  dist.Normal(0.0, 0.3))
    b_off_sack_rate_allowed = numpyro.sample("b_off_sack_rate_allowed", dist.Normal(0.0, 0.2))
    b_def_sack_rate         = numpyro.sample("b_def_sack_rate",         dist.Normal(0.0, 0.2))
    # Archetype embeddings: 4-vector per archetype type (int32 index 0-3)
    # Do NOT use 16-way compound matchup string columns
    b_off_archetype         = numpyro.sample("b_off_archetype",
                                              dist.Normal(0.0, 0.3),
                                              sample_shape=(4,))
    b_def_archetype         = numpyro.sample("b_def_archetype",
                                              dist.Normal(0.0, 0.3),
                                              sample_shape=(4,))
    b_days_since_last_game  = numpyro.sample("b_days_since_last_game",  dist.Normal(0.0, 0.2))


# -----------------------------------------------------------------------------
# Verify: draw one prior sample — confirms all distributions are valid
# No MCMC, no data, no fitting
# -----------------------------------------------------------------------------
from jax import random

rng_key     = random.PRNGKey(42)
prior_trace = numpyro.infer.Predictive(model_prior_spec, num_samples=1)(rng_key)

print("Prior specification verified — all parameters sampled successfully:")
print(f"  Total parameters sampled : {len(prior_trace)}")
print()
for name, val in sorted(prior_trace.items()):
    print(f"  {name:<30s} shape={str(val.shape):<20s} sample={float(val.flatten()[0]):.4f}")

print("\nCell 6 complete. model_01 prior specification is done.")

Prior specification verified — all parameters sampled successfully:
  Total parameters sampled : 34

  alpha_team                     shape=(1, 130)             sample=-0.3360
  b_away_elevation_delta         shape=(1,)                 sample=-0.2253
  b_away_travel_distance         shape=(1,)                 sample=0.0402
  b_away_tz_delta                shape=(1,)                 sample=0.0419
  b_close_game_def_epa           shape=(1,)                 sample=0.4686
  b_close_game_epa               shape=(1,)                 sample=0.2654
  b_days_since_last_game         shape=(1,)                 sample=-0.0211
  b_def_archetype                shape=(1, 4)               sample=0.2843
  b_def_sack_rate                shape=(1,)                 sample=0.0451
  b_elo_sp_divergence            shape=(1,)                 sample=-0.1058
  b_last3_def_epa_avg            shape=(1,)                 sample=-0.0299
  b_last3_off_epa_avg            shape=(1,)                 sample=0.1958
  b_la

## Notebook Summary — model_01_prior_specification.ipynb

### What This Notebook Does

This notebook translates the prior specification document (`artifacts/prior_specification_draft.md`) into NumPyro code. It defines every probability distribution the model will use. It does not fit the model, does not touch any data, and does not run MCMC.

### Why This Exists as a Separate Notebook

The model has over 130 parameters across three levels of hierarchy. Every one of those parameters needs a prior distribution — a statement of what values are plausible before the model sees any game data. Those decisions were made during EDA and recorded in the prior specification document. This notebook's only job is to translate those decisions into code faithfully, with citations, so that nothing is invented or assumed when the model is actually built in model_02 and fit in model_04.

### What a Prior Distribution Is

A prior distribution expresses what the model believes about a parameter before seeing data. For example, the league scoring intercept is given `Normal(3.3, 0.2)` on the log scale — `exp(3.3) ≈ 27 pts`, the FBS mean from 2022–2024. The SD of 0.2 allows the posterior to move freely while ruling out absurd values. After fitting, the data updates these beliefs into posterior distributions.

### The Three-Level Hierarchy

The model pools information across three levels:

- **League** — one baseline for all of FBS: intercept, home field advantage, and a conference-specific dispersion vector (one `r_negbinom` per conference)
- **Conference** — ten conference offsets from the league baseline, sharing a common hyperprior
- **Team** — individual attack, defense, and home field parameters for each of the ~130 FBS teams, pooled within their conference

Pooling means that teams with little data borrow strength from their conference, and conferences borrow from the league. This prevents overfitting on small samples.

### Conference-Specific Dispersion

The negative binomial dispersion parameter `r_negbinom` is a vector of length 10 — one value per conference. EDA 05 confirmed statistically significant dispersion differences across conferences (Bartlett p=0.000470, Levene p=0.000705). A single scalar `r` cannot capture this variation and caused sampler collapse in earlier model versions. Each conference's `r` is given an independent `Gamma(2.0, 0.1)` prior.

### The Sun Belt Constraint

The recruiting weight for the Sun Belt conference is hard-constrained to be non-positive. EDA 04 found that recruiting composite scores are negatively correlated with point differential in the Sun Belt (r = -0.2665). Treating recruiting as a positive signal there would introduce systematic error. This is implemented as `TruncatedNormal(high=0.0)` — the distribution is physically incapable of producing a positive value.

### Archetype Encoding Contract

Style archetypes (EDA 10) are 4-level categorical variables, not continuous scores. Each is embedded as a 4-vector: `b_off_archetype` and `b_def_archetype`, each with `sample_shape=(4,)`. In model_02, these are indexed as `b_off_archetype[data.off_archetype_idx]` where `data.off_archetype_idx` is an `int32` array with values 0–3. The 16-way compound matchup string columns must not be used.

### What the Next Notebook Does

`model_02_architecture.ipynb` takes these prior definitions and builds the full model function — adding the likelihood (Negative Binomial), the log-linear scoring equation, and the conference-scope masking for game-level features. `model_04_first_fit.ipynb` fits that model on 2022–2024 training data for the first time.